# 02 — Young (2025) QISICGM: phase feature map + RF/ET/MLP ensemble, GMM augmentation

**Method (arXiv 2509.12259):** standardise, expand each feature x into (cos x, sin x)
("quantum-inspired" phase map), average the probabilities of a random forest, extra trees and an
MLP. The paper added 2,000 GMM-synthetic diabetic rows to Pima's 768 **before** 5-fold CV, so
synthetic rows also sat in the validation folds.

**Protocols**
* `paper` — synthetic positives (GMM, 5 components, fitted on all real positives) added to the
  dataset before ungrouped stratified 5-fold CV (seed 42); scored on all validation rows,
  synthetic included. Synthetic count = 2000/768 × N (capped at 5,000).
* `paper_real_only` — same augmented training, but scored on real validation rows only.
* `clean` — no synthetic rows in any test fold: GMM fitted on the training-fold positives only and
  used to augment the training fold only; grouped/stratified 5-fold × seeds {0, 1, 42}.
  The same ensemble without augmentation is also reported.

In [1]:
import os, sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd() / "notebooks2", Path("/content/notebooks2"), *Path("/kaggle/input").glob("*"), *Path("/kaggle/input").glob("*/notebooks2")]:
    if (p / "nb2_common.py").exists():
        sys.path.insert(0, str(p)); break
from nb2_common import *
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier
sanity_check_kernels()
DATA = load_all()

nb2_common loaded: numpy 2.4.6, sklearn 1.6.1
kernel self-test passed
pima     N=  768 d=  8 prevalence=0.349 unique=  768 grouped_cv=False  [diabetes]


sylhet   N=  520 d= 16 prevalence=0.615 unique=  251 grouped_cv=True  [diabetes (early stage)]


cdc      N= 2000 d= 21 prevalence=0.140 unique= 1995 grouped_cv=True  [diabetes or prediabetes (self-reported)]


hosp130  N= 2000 d= 44 prevalence=0.088 unique= 2000 grouped_cv=False  [readmitted within 30 days]


In [2]:
def phase_map(X):
    return np.hstack([np.cos(X), np.sin(X)])


def ensemble(Ztr, ytr, Zte, seed):
    P = [RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1).fit(Ztr, ytr).predict_proba(Zte)[:, 1],
         ExtraTreesClassifier(n_estimators=100, random_state=seed, n_jobs=-1).fit(Ztr, ytr).predict_proba(Zte)[:, 1],
         MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=seed).fit(Ztr, ytr).predict_proba(Zte)[:, 1]]
    p = np.mean(P, axis=0)
    return (p >= 0.5).astype(int), p


def n_synth(n):
    return int(min(5000, round(2000 / 768 * n)))


def gmm_positives(S_pos, n, seed):
    g = GaussianMixture(n_components=min(5, max(1, len(S_pos) // 10)), covariance_type="full",
                        reg_covar=1e-3, random_state=seed).fit(S_pos)
    return g.sample(n)[0]


def clean_method(Xtr, ytr, Xte, seed, groups_tr, X_prep):
    Sr, Se = prep(impute_scale, X_prep, Xtr, Xte)
    out = {"QISICGM ensemble (no augmentation)": ensemble(phase_map(Sr), ytr, phase_map(Se), seed)}
    syn = gmm_positives(Sr[ytr == 1], n_synth(len(ytr)), seed)
    Sa = np.vstack([Sr, syn]); ya = np.r_[ytr, np.ones(len(syn), int)]
    out["QISICGM ensemble (+GMM, train-fold only)"] = ensemble(phase_map(Sa), ya, phase_map(Se), seed)
    return out


def run_paper(rec, seed=PAPER_SEED):
    for name in DATASETS:
        ds = DATA[name]
        S = impute_scale(ds["X"])[0]                          # global scaling (paper)
        syn = gmm_positives(S[ds["y"] == 1], n_synth(len(ds["y"])), seed)
        SA = np.vstack([S, syn]); yA = np.r_[ds["y"], np.ones(len(syn), int)]
        is_real = np.r_[np.ones(len(S), bool), np.zeros(len(syn), bool)]
        for k, (tr, te) in enumerate(StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed).split(SA, yA)):
            pred, p = ensemble(phase_map(SA[tr]), yA[tr], phase_map(SA[te]), seed)
            rec.add(name, "paper", seed, k, "QISICGM ensemble (+GMM, global)", yA[te], pred, p,
                    synthetic_in_test=int((~is_real[te]).sum()))
            r = is_real[te]
            rec.add(name, "paper_real_only", seed, k, "QISICGM ensemble (+GMM, global)", yA[te][r], pred[r], p[r])
        print(f"  paper  {name:8s} done ({len(syn)} synthetic positives)")


rec = Recorder("P02_Young", "02_young", config=dict(
    method="phase map [cos x, sin x] -> mean prob of RF(100), ET(100), MLP(32,16); GMM(<=5 comp) synthetic positives",
    synthetic_ratio="2000/768 x N, cap 5000"))
print("paper protocol"); run_paper(rec)
print("clean protocol"); run_clean(rec, DATA, clean_method)
df = rec.save()

paper protocol


  paper  pima     done (2000 synthetic positives)


  paper  sylhet   done (1354 synthetic positives)


  paper  cdc      done (5000 synthetic positives)


  paper  hosp130  done (5000 synthetic positives)
clean protocol


  clean  pima     done in   69.1s


  clean  sylhet   done in   37.8s


  clean  cdc      done in  121.8s


  clean  hosp130  done in  136.0s
saved /content/notebooks2/results/02_young.csv  (160 rows)


In [3]:
pd.set_option("display.width", 220, "display.max_rows", 200)
summarise(df, ("accuracy", "f1", "recall", "specificity", "roc_auc"))

,dataset,protocol,model,accuracy,f1,recall,specificity,roc_auc,n_runs
0,cdc,clean,"QISICGM ensemble (+GMM, train-fold only)",0.85 ± 0.015,0.336 ± 0.056,0.276 ± 0.054,0.943 ± 0.014,0.784 ± 0.02,15
1,cdc,clean,QISICGM ensemble (no augmentation),0.858 ± 0.012,0.251 ± 0.059,0.175 ± 0.053,0.97 ± 0.015,0.798 ± 0.029,15
2,cdc,paper,"QISICGM ensemble (+GMM, global)",0.957 ± 0.003,0.972 ± 0.002,0.963 ± 0.002,0.94 ± 0.007,0.989 ± 0.001,5
3,cdc,paper_real_only,"QISICGM ensemble (+GMM, global)",0.851 ± 0.01,0.363 ± 0.024,0.304 ± 0.019,0.94 ± 0.007,0.787 ± 0.022,5
4,hosp130,clean,"QISICGM ensemble (+GMM, train-fold only)",0.9 ± 0.006,0.016 ± 0.023,0.01 ± 0.014,0.986 ± 0.006,0.548 ± 0.052,15
5,hosp130,clean,QISICGM ensemble (no augmentation),0.906 ± 0.005,0.01 ± 0.02,0.006 ± 0.012,0.993 ± 0.005,0.548 ± 0.044,15
6,hosp130,paper,"QISICGM ensemble (+GMM, global)",0.972 ± 0.005,0.981 ± 0.003,0.967 ± 0.007,0.986 ± 0.005,0.985 ± 0.004,5
7,hosp130,paper_real_only,"QISICGM ensemble (+GMM, global)",0.902 ± 0.016,0.041 ± 0.068,0.026 ± 0.044,0.986 ± 0.005,0.576 ± 0.055,5
8,pima,clean,"QISICGM ensemble (+GMM, train-fold only)",0.663 ± 0.033,0.645 ± 0.025,0.876 ± 0.041,0.549 ± 0.056,0.809 ± 0.03,15
9,pima,clean,QISICGM ensemble (no augmentation),0.771 ± 0.025,0.651 ± 0.044,0.616 ± 0.068,0.855 ± 0.032,0.827 ± 0.03,15
